# EDUCAÇÃO — ABANDONO ESCOLAR E TRABALHO JUVENIL

## Checkpoint 2 — Modelagem Preditiva

## Integrantes

* **Ana Luiza Santos Bertão** — RM 563171
* **Anny Elly Pantoja Martin** — RM 565055
* **Gisleine Muñoz Ticona** — RM 563804
* **Larissa Machado Silva** — RM 564168
* **Raira de Teves Costa** — RM 564850
* **Sofia Bill Franken** — RM 562767

## 1. Retomada da Parte 1

Na Parte 1 analisamos a **relação entre o abandono escolar e as condições de inserção dos jovens no mercado de trabalho no Brasil**, usando duas bases oficiais:

- **INEP** — Taxas de Rendimento Escolar (2013–2023): taxas de abandono no Ensino Fundamental e no Ensino Médio.
- **IBGE / SIDRA, Tabela 4094** — taxa de desocupação (código 4099) e taxa de informalidade (código 12466) dos jovens de 14 a 17 e de 18 a 24 anos, por Grande Região.

**Pergunta inicial:** o abandono escolar se associa aos indicadores de desocupação e informalidade juvenil nas regiões brasileiras?

Na Parte 1 a resposta foi **descritiva** (correlações e gráficos). Nesta Parte 2 damos o passo seguinte: verificar se esses indicadores conseguem **prever** o abandono escolar.

As células abaixo reproduzem, sem alterações de lógica, o carregamento, o tratamento e a integração das bases feitos na Parte 1, para que este notebook rode de forma independente.

## 2. Bibliotecas

In [ ]:
# Se estiver no Google Colab ou sem as bibliotecas instaladas, descomente a linha abaixo:
# !pip install -q xgboost tensorflow

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (10, 5)

## 3. Carregamento e tratamento das bases (Parte 1)

In [ ]:
caminho_inep = Path('../dados/INEP/Taxas_de_Rendimento_Escolar_2013_2023.csv')
caminho_ibge = Path('../dados/IBGE/IBGE_Trabalho_Juvenil_2013_2025.csv')

if not caminho_inep.exists() or not caminho_ibge.exists():
    raise FileNotFoundError(
        'Não foi possível localizar uma ou ambas as bases. Verifique se os arquivos estão em ../dados/INEP e ../dados/IBGE.'
    )

df_inep = pd.read_csv(caminho_inep, encoding='utf-8')
df_ibge = pd.read_csv(caminho_ibge, encoding='utf-8-sig')

print('Dimensões do INEP:', df_inep.shape)
print('Dimensões do IBGE:', df_ibge.shape)

In [ ]:
# Tratamento (igual à Parte 1): remove apenas duplicidades exatas e converte os tipos
df_inep_tratado = df_inep.copy().drop_duplicates()
df_ibge_tratado = df_ibge.copy().drop_duplicates()

df_inep_tratado['Ano'] = pd.to_numeric(df_inep_tratado['Ano'], errors='coerce').astype('Int64')
for col in ['Taxa_Abandono_Total_EF', 'Taxa_Abandono_Total_EM']:
    df_inep_tratado[col] = pd.to_numeric(df_inep_tratado[col], errors='coerce')

df_ibge_tratado['V'] = pd.to_numeric(df_ibge_tratado['V'], errors='coerce')
df_ibge_tratado['D2C'] = pd.to_numeric(df_ibge_tratado['D2C'], errors='coerce').astype('Int64')
df_ibge_tratado['Ano'] = pd.to_numeric(df_ibge_tratado['D3C'].astype(str).str[:4], errors='coerce').astype('Int64')

# INEP: apenas recortes "Total" para não duplicar ano/região
inep_kpi = df_inep_tratado.copy()
inep_kpi = inep_kpi[inep_kpi['Localizacao'].astype(str).str.strip().str.lower().eq('total')]
inep_kpi = inep_kpi[inep_kpi['Dependencia_Administrativa'].astype(str).str.strip().str.lower().eq('total')]
inep_kpi = inep_kpi.dropna(subset=['Ano', 'Unidade_Geografica'])
inep_kpi = inep_kpi[['Ano', 'Unidade_Geografica', 'Taxa_Abandono_Total_EF', 'Taxa_Abandono_Total_EM']].copy()
inep_kpi = inep_kpi.rename(columns={'Unidade_Geografica': 'Regiao'})

# IBGE: desocupação (4099) e informalidade (12466)
ibge_indicadores = df_ibge_tratado[df_ibge_tratado['D2C'].isin([4099, 12466])].copy()
ibge_indicadores = ibge_indicadores.dropna(subset=['Ano', 'D1N', 'D4N', 'V'])
ibge_indicadores['Indicador'] = ibge_indicadores['D2C'].map({4099: 'Taxa de desocupação', 12466: 'Taxa de informalidade'})

# Integração por Ano + Região (média entre as faixas etárias do IBGE)
ibge_regiao_ano = ibge_indicadores.groupby(['Ano', 'D1N', 'Indicador'], as_index=False)['V'].mean()
ibge_largo = ibge_regiao_ano.pivot_table(index=['Ano', 'D1N'], columns='Indicador', values='V').reset_index()
ibge_largo = ibge_largo.rename(columns={'D1N': 'Regiao'})

integrado = inep_kpi.merge(ibge_largo, on=['Ano', 'Regiao'], how='inner')

print('Dimensão da base integrada (Parte 1):', integrado.shape)
print('Anos disponíveis:', sorted(integrado['Ano'].astype(int).unique().tolist()))
print('Regiões:', sorted(integrado['Regiao'].unique().tolist()))
integrado.head()

## 4. Definição do problema de Machine Learning

### 4.1 Objetivo da análise e tipo de problema

**Objetivo:** construir modelos que **prevejam a taxa de abandono escolar no Ensino Médio de uma Grande Região, em um dado ano, a partir dos indicadores de mercado de trabalho juvenil (desocupação e informalidade) e da própria região**, e avaliar se esses indicadores têm poder preditivo real, indo além da correlação analisada na Parte 1.

**Tipo de problema:** **regressão supervisionada**, pois a variável a prever (uma taxa em %) é contínua.

### 4.2 Variável alvo e variáveis preditoras

| Papel | Variável | Descrição |
|---|---|---|
| **Target** | `abandono_em` | Taxa de abandono total no Ensino Médio (%) — INEP |
| Feature | `desocupacao_juvenil` | Taxa de desocupação juvenil (%) — IBGE (média das faixas 14–17 e 18–24 anos) |
| Feature | `informalidade_juvenil` | Taxa de informalidade juvenil (%) — IBGE (média das faixas 14–17 e 18–24 anos) |
| Feature | `Regiao` | Grande Região (Norte, Nordeste, Sudeste, Sul, Centro-Oeste) — categórica |

**Por que o Ensino Médio como alvo?** Na Parte 1 o Ensino Médio apresentou as maiores taxas de abandono e é a etapa diretamente ligada à transição para a vida profissional, exatamente o ponto da pergunta inicial.

**Por que a taxa do Ensino Fundamental não entra como feature?** Ela é outro desfecho do mesmo fenômeno (correlação de 0,85 com o abandono no EM na Parte 1). Incluí-la faria o modelo "prever abandono com abandono" e esconderia a contribuição dos indicadores de trabalho, que são o foco da pergunta.

**Por que `Ano` não entra como feature?** Para que o modelo não aprenda apenas a tendência temporal de queda do abandono. Ele precisa explicar o abandono pelos indicadores de trabalho e pela região. O ano é usado apenas para dividir treino e teste (seção 5.3).

### 4.3 Justificativa e relevância

- **Continuidade com a Parte 1:** a Parte 1 mostrou associações (correlação de 0,48 entre abandono no EM e informalidade; −0,33 com desocupação). Correlação não é capacidade de previsão. Aqui testamos se os indicadores realmente reduzem o erro de previsão frente a um modelo ingênuo.
- **Apoio à decisão:** um modelo que estime o abandono esperado de cada região a partir de indicadores de trabalho que **saem com frequência trimestral** (PNAD Contínua) ajuda a sinalizar regiões em risco, priorizar ações de permanência escolar e de qualificação profissional.
- **Honestidade sobre os limites:** os dados são agregados (região × ano), o que gera poucas observações e impede conclusões sobre indivíduos ou causalidade. Os resultados devem ser lidos como **evidência preditiva em nível agregado**.

In [ ]:
# Base de modelagem: nomes de colunas padronizados (sem espaços/acentos)
df_modelo = integrado.rename(columns={
    'Taxa_Abandono_Total_EM': 'abandono_em',
    'Taxa_Abandono_Total_EF': 'abandono_ef',
    'Taxa de desocupação': 'desocupacao_juvenil',
    'Taxa de informalidade': 'informalidade_juvenil',
}).copy()
df_modelo['Ano'] = df_modelo['Ano'].astype(int)

TARGET = 'abandono_em'
FEATURES_NUM = ['desocupacao_juvenil', 'informalidade_juvenil']
FEATURES_CAT = ['Regiao']
FEATURES = FEATURES_NUM + FEATURES_CAT

print('Observações (região × ano):', len(df_modelo))
df_modelo[['Ano', 'Regiao', TARGET] + FEATURES_NUM].sort_values(['Ano', 'Regiao']).head(10)

## 5. Preparação dos dados

### 5.1 Etapas de pré-processamento

1. **Filtragem e integração (Parte 1):** apenas recortes `Total` de localização e dependência administrativa no INEP; integração com o IBGE por **Ano + Grande Região**; média das faixas etárias do IBGE.
2. **Conversão de tipos:** colunas numéricas convertidas com `pd.to_numeric`; registros de cabeçalho repetido da API do SIDRA descartados.
3. **Valores ausentes:** a informalidade não existe nos primeiros anos da série do IBGE. Em vez de descartar linhas (já temos poucas), os ausentes são **imputados pela mediana**, calculada **somente no treino** para evitar vazamento de dados.
4. **Variável categórica:** `Regiao` transformada em colunas binárias (**One-Hot Encoding**).
5. **Padronização:** variáveis numéricas padronizadas (`StandardScaler`). É essencial para a rede neural e inofensiva para os modelos de árvore, então o mesmo pré-processamento vale para os três modelos.
6. **Sem vazamento:** imputação, padronização e codificação ficam dentro de um `Pipeline`/`ColumnTransformer`, ajustados apenas com os dados de treino.

### 5.2 Verificação dos dados

In [ ]:
print('Valores ausentes por coluna:')
print(df_modelo[FEATURES + [TARGET]].isna().sum())

print('\nValores ausentes de informalidade por ano:')
print(df_modelo.groupby('Ano')['informalidade_juvenil'].apply(lambda s: s.isna().sum()))

assert df_modelo[TARGET].notna().all(), 'O target não pode ter valores ausentes.'
df_modelo[FEATURES_NUM + [TARGET]].describe().round(2)

### 5.3 Separação entre features e target e entre treino e teste

Os dados formam uma **série temporal por região**. Um sorteio aleatório colocaria, no treino, regiões de anos vizinhos ao teste e deixaria o modelo "espiar o futuro". Por isso usamos uma **divisão temporal**:

- **Treino:** anos até 2019
- **Teste:** anos a partir de 2022 (período pós-pandemia, com dinâmica diferente — um teste mais exigente e mais realista)

Isso simula o uso real do modelo: aprender com o passado e prever períodos novos. O teste é pequeno (poucas regiões × poucos anos), o que é uma limitação discutida na avaliação.

In [ ]:
ANO_CORTE = 2019

mascara_treino = df_modelo['Ano'] <= ANO_CORTE
X = df_modelo[FEATURES]
y = df_modelo[TARGET]

X_train, y_train = X[mascara_treino].copy(), y[mascara_treino].copy()
X_test,  y_test  = X[~mascara_treino].copy(), y[~mascara_treino].copy()

assert len(X_test) > 0 and len(X_train) > 0, 'Divisão temporal vazia: ajuste ANO_CORTE.'

print(f'Treino: {len(X_train)} linhas | anos {sorted(df_modelo.loc[mascara_treino, "Ano"].unique().tolist())}')
print(f'Teste : {len(X_test)} linhas | anos {sorted(df_modelo.loc[~mascara_treino, "Ano"].unique().tolist())}')
print(f'Proporção treino/teste: {len(X_train)/len(X):.0%} / {len(X_test)/len(X):.0%}')

In [ ]:
def make_preprocessor():
    # Imputação (mediana) + padronização nas numéricas; One-Hot na região
    numericas = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])
    return ColumnTransformer([
        ('num', numericas, FEATURES_NUM),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), FEATURES_CAT),
    ])

cv = KFold(n_splits=5, shuffle=True, random_state=SEED)  # validação cruzada dentro do treino


def calcular_metricas(y_real, y_prev):
    return {
        'MAE': mean_absolute_error(y_real, y_prev),
        'RMSE': float(np.sqrt(mean_squared_error(y_real, y_prev))),
        'R²': r2_score(y_real, y_prev),
    }

preds_train, preds_test = {}, {}

## 6. Treinamento dos modelos

**Métricas** (todas em pontos percentuais de abandono, exceto o R²):
- **MAE:** erro absoluto médio, fácil de interpretar;
- **RMSE:** penaliza mais os erros grandes (métrica usada para escolher hiperparâmetros);
- **R²:** quanto da variação do abandono o modelo explica. Valores ≤ 0 indicam que o modelo não supera simplesmente prever a média.

Antes dos três modelos, criamos um **baseline**, que sempre prevê a média do abandono no treino. Qualquer modelo útil precisa superá-lo.

### 6.0 Baseline (referência)

In [ ]:
baseline = DummyRegressor(strategy='mean').fit(X_train, y_train)
preds_train['Baseline (média)'] = baseline.predict(X_train)
preds_test['Baseline (média)'] = baseline.predict(X_test)
print('Previsão constante do baseline: %.2f%%' % baseline.constant_[0, 0] if np.ndim(baseline.constant_) == 2 else 'Previsão constante do baseline: %.2f%%' % float(baseline.constant_))

### 6.1 Modelo 1 — Random Forest (scikit-learn)

Conjunto de várias árvores de decisão treinadas em amostras diferentes, com média das previsões. Lida bem com relações não lineares e é robusto com poucos dados. Os hiperparâmetros são escolhidos por **GridSearchCV** com validação cruzada de 5 partições, apenas no treino.

In [ ]:
pipe_rf = Pipeline([
    ('prep', make_preprocessor()),
    ('model', RandomForestRegressor(random_state=SEED)),
])

grid_rf = {
    'model__n_estimators': [200, 500],
    'model__max_depth': [2, 3, None],
    'model__min_samples_leaf': [1, 2, 4],
}

busca_rf = GridSearchCV(pipe_rf, grid_rf, cv=cv, scoring='neg_root_mean_squared_error', n_jobs=-1)
busca_rf.fit(X_train, y_train)

modelo_rf = busca_rf.best_estimator_
cv_rmse_rf = -busca_rf.best_score_

print('Melhores hiperparâmetros:', busca_rf.best_params_)
print(f'RMSE médio na validação cruzada (treino): {cv_rmse_rf:.3f}')

preds_train['Random Forest'] = modelo_rf.predict(X_train)
preds_test['Random Forest'] = modelo_rf.predict(X_test)

### 6.2 Modelo 2 — XGBoost (árvores com *gradient boosting*)

Em vez de árvores independentes, o XGBoost treina árvores **em sequência**, cada uma corrigindo os erros da anterior. Costuma ser muito competitivo em dados tabulares. Com poucas observações, usamos árvores rasas e taxa de aprendizado baixa para reduzir o risco de *overfitting*.

In [ ]:
pipe_xgb = Pipeline([
    ('prep', make_preprocessor()),
    ('model', XGBRegressor(objective='reg:squarederror', random_state=SEED, n_jobs=1)),
])

grid_xgb = {
    'model__n_estimators': [100, 300],
    'model__max_depth': [2, 3],
    'model__learning_rate': [0.03, 0.1],
    'model__subsample': [0.8, 1.0],
}

busca_xgb = GridSearchCV(pipe_xgb, grid_xgb, cv=cv, scoring='neg_root_mean_squared_error', n_jobs=-1)
busca_xgb.fit(X_train, y_train)

modelo_xgb = busca_xgb.best_estimator_
cv_rmse_xgb = -busca_xgb.best_score_

print('Melhores hiperparâmetros:', busca_xgb.best_params_)
print(f'RMSE médio na validação cruzada (treino): {cv_rmse_xgb:.3f}')

preds_train['XGBoost'] = modelo_xgb.predict(X_train)
preds_test['XGBoost'] = modelo_xgb.predict(X_test)

### 6.3 Modelo 3 — Rede Neural Profunda (Keras / TensorFlow)

Rede *feed-forward* com **3 camadas ocultas** (64 → 32 → 16 neurônios, ativação ReLU), regularização L2 e *Dropout* para conter o *overfitting*, e saída linear (regressão). O treino usa **Early Stopping**: ele para quando o erro de validação deixa de melhorar e restaura os melhores pesos.

> Com cerca de 35 linhas de treino, uma rede profunda **não é a escolha mais adequada**: ela tende a ser instável e a perder para modelos mais simples. Ela entra aqui por exigência do enunciado e como contraponto na comparação.

O alvo também é padronizado para estabilizar o treino, e as previsões voltam para a escala original (%) antes de calcular as métricas.

In [ ]:
# Pré-processamento ajustado só no treino
prep_nn = make_preprocessor()
X_train_nn = prep_nn.fit_transform(X_train)
X_test_nn = prep_nn.transform(X_test)

scaler_y = StandardScaler()
y_train_nn = scaler_y.fit_transform(y_train.values.reshape(-1, 1))

# Validação interna (20% do treino) para o Early Stopping
X_tr, X_val, y_tr, y_val = train_test_split(X_train_nn, y_train_nn, test_size=0.2, random_state=SEED)

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

modelo_nn = keras.Sequential([
    layers.Input(shape=(X_train_nn.shape[1],)),
    layers.Dense(64, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-3)),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-3)),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(1),
])
modelo_nn.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss='mse', metrics=['mae'])
modelo_nn.summary()

early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=40, restore_best_weights=True)

historico = modelo_nn.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=500, batch_size=8,
    callbacks=[early_stop], verbose=0,
)
print('Épocas treinadas:', len(historico.history['loss']))

In [ ]:
plt.figure()
plt.plot(historico.history['loss'], label='Treino')
plt.plot(historico.history['val_loss'], label='Validação')
plt.title('Rede neural — curva de aprendizado (perda MSE, alvo padronizado)')
plt.xlabel('Época')
plt.ylabel('Perda')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

preds_train['Rede Neural'] = scaler_y.inverse_transform(modelo_nn.predict(X_train_nn, verbose=0)).ravel()
preds_test['Rede Neural'] = scaler_y.inverse_transform(modelo_nn.predict(X_test_nn, verbose=0)).ravel()

## 7. Avaliação e comparação dos modelos

In [ ]:
ordem = ['Baseline (média)', 'Random Forest', 'XGBoost', 'Rede Neural']

tabela_teste = pd.DataFrame({m: calcular_metricas(y_test, preds_test[m]) for m in ordem}).T
tabela_treino = pd.DataFrame({m: calcular_metricas(y_train, preds_train[m]) for m in ordem}).T

print('=== Desempenho no TESTE (anos não vistos) ===')
display(tabela_teste.round(3))
print('=== Desempenho no TREINO ===')
display(tabela_treino.round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
cores = ['#9e9e9e', '#2e7d32', '#1565c0', '#ef6c00']
axes[0].bar(ordem, tabela_teste['RMSE'], color=cores)
axes[0].set_title('RMSE no teste (menor é melhor)')
axes[0].set_ylabel('p.p. de abandono')
axes[1].bar(ordem, tabela_teste['MAE'], color=cores)
axes[1].set_title('MAE no teste (menor é melhor)')
for ax in axes:
    ax.tick_params(axis='x', rotation=20)
    ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Previsto x observado (teste) — pontos na diagonal = previsão perfeita
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True)
lim = [min(y_test.min(), *[preds_test[m].min() for m in ordem[1:]]) - 0.5,
       max(y_test.max(), *[preds_test[m].max() for m in ordem[1:]]) + 0.5]
for ax, m in zip(axes, ordem[1:]):
    ax.scatter(y_test, preds_test[m], s=55, alpha=0.8)
    ax.plot(lim, lim, 'k--', alpha=0.5)
    ax.set_title(m)
    ax.set_xlabel('Abandono EM observado (%)')
    ax.grid(alpha=0.25)
axes[0].set_ylabel('Abandono EM previsto (%)')
plt.tight_layout()
plt.show()

### 7.1 Estabilidade: desempenho no teste x validação cruzada

Com um teste tão pequeno, olhar só para ele é arriscado. Comparamos o RMSE do teste com o da validação cruzada no treino (Random Forest e XGBoost): se forem muito diferentes, o resultado do teste pode ser fruto do acaso ou de uma mudança de padrão entre os períodos.

In [ ]:
estabilidade = pd.DataFrame({
    'RMSE validação cruzada (treino)': [cv_rmse_rf, cv_rmse_xgb],
    'RMSE teste': [tabela_teste.loc['Random Forest', 'RMSE'], tabela_teste.loc['XGBoost', 'RMSE']],
}, index=['Random Forest', 'XGBoost'])
display(estabilidade.round(3))

### 7.2 Quais indicadores mais contribuíram para as previsões?

Usamos a **importância por permutação**: embaralhamos uma variável por vez no conjunto de teste e medimos o quanto o erro (RMSE) piora. Quanto maior a piora, mais o modelo depende daquela variável.

In [ ]:
def importancia(modelo):
    r = permutation_importance(modelo, X_test, y_test, n_repeats=30, random_state=SEED,
                               scoring='neg_root_mean_squared_error')
    return pd.Series(r.importances_mean, index=FEATURES)

importancias = pd.DataFrame({
    'Random Forest': importancia(modelo_rf),
    'XGBoost': importancia(modelo_xgb),
})
display(importancias.round(3))

importancias.plot(kind='barh', figsize=(9, 4))
plt.title('Importância por permutação — aumento do RMSE ao embaralhar a variável')
plt.xlabel('Aumento do RMSE (p.p.)')
plt.grid(axis='x', alpha=0.25)
plt.show()

# Correlações da Parte 1 (base completa), para comparação
corr_parte1 = df_modelo[[TARGET] + FEATURES_NUM].corr()[TARGET].drop(TARGET)
print('Correlação de Pearson com o abandono no EM (Parte 1):')
print(corr_parte1.round(2))

In [ ]:
# Previsão x realidade por região no período de teste (melhor modelo entre os três)
modelos_ml = ordem[1:]
melhor = tabela_teste.loc[modelos_ml, 'RMSE'].idxmin()

detalhe = df_modelo.loc[~mascara_treino, ['Ano', 'Regiao', TARGET]].copy()
detalhe['previsto'] = preds_test[melhor]
detalhe['erro'] = detalhe['previsto'] - detalhe[TARGET]
detalhe = detalhe.rename(columns={TARGET: 'observado'}).sort_values(['Ano', 'Regiao'])
print(f'Melhor modelo (menor RMSE no teste): {melhor}')
display(detalhe.round(2))

## 8. Interpretação dos resultados

### 8.1 Leitura automática dos resultados desta execução

A célula abaixo escreve as conclusões numéricas a partir das métricas calculadas, sem valores digitados à mão.

In [ ]:
rmse_base = tabela_teste.loc['Baseline (média)', 'RMSE']
rmse_melhor = tabela_teste.loc[melhor, 'RMSE']
ganho = (1 - rmse_melhor / rmse_base) * 100

print(f'• Melhor modelo no teste: {melhor} (RMSE = {rmse_melhor:.2f} p.p.; '
      f'MAE = {tabela_teste.loc[melhor, "MAE"]:.2f} p.p.; R² = {tabela_teste.loc[melhor, "R²"]:.2f}).')
print(f'• Baseline (prever a média): RMSE = {rmse_base:.2f} p.p. '
      f'-> o melhor modelo {"reduziu" if ganho > 0 else "aumentou"} o erro em {abs(ganho):.1f}% frente ao baseline.')

for m in modelos_ml:
    r2 = tabela_teste.loc[m, 'R²']
    status = 'supera' if tabela_teste.loc[m, 'RMSE'] < rmse_base else 'NÃO supera'
    gap = tabela_teste.loc[m, 'RMSE'] - tabela_treino.loc[m, 'RMSE']
    print(f'  - {m}: {status} o baseline (R² teste = {r2:.2f}); '
          f'RMSE teste - treino = {gap:+.2f} p.p. ({"possível overfitting" if gap > 0.5 * max(tabela_treino.loc[m, "RMSE"], 1e-9) else "diferença moderada"}).')

top_rf = importancias['Random Forest'].idxmax()
top_xgb = importancias['XGBoost'].idxmax()
print(f'• Variável mais importante — Random Forest: {top_rf}; XGBoost: {top_xgb}.')
print('• Na Parte 1, a maior correlação com o abandono no EM foi com: '
      f'{corr_parte1.abs().idxmax()} (r = {corr_parte1[corr_parte1.abs().idxmax()]:.2f}).')

### 8.2 Como os resultados complementam os indicadores da Parte 1

| | Parte 1 (descritiva) | Parte 2 (preditiva) |
|---|---|---|
| **Pergunta** | O que aconteceu e como os indicadores se relacionam? | Os indicadores de trabalho conseguem **prever** o abandono em períodos novos? |
| **Ferramenta** | KPIs, gráficos e correlação de Pearson | Modelos de regressão avaliados em anos não vistos |
| **Evidência** | Associação linear média entre os indicadores | Redução (ou não) do erro de previsão em relação a um modelo que só prevê a média |

**Como ler o resultado:**

- Se os modelos **superam o baseline** no teste, os indicadores de trabalho e a região carregam **informação útil** sobre o abandono, o que reforça, de forma preditiva, a associação encontrada na Parte 1. Isso sustenta o uso desses indicadores como **sinais de alerta** para priorizar regiões.
- Se os modelos **não superam** o baseline ou o desempenho é instável entre validação cruzada e teste, a conclusão é igualmente relevante: a correlação da Parte 1 **não se traduz em capacidade de previsão confiável** com esses dados, e o abandono depende de fatores não incluídos (renda, infraestrutura escolar, políticas, efeitos da pandemia).
- A **importância por permutação** mostra quais indicadores o modelo realmente usa e permite compará-los com as correlações da Parte 1. Um indicador com correlação moderada pode ter importância preditiva diferente, porque o modelo captura relações **não lineares** e diferenças entre regiões.
- A **análise por região** mostra onde o modelo erra mais, indicando onde o abandono é menos explicado pelos indicadores de trabalho.

> **Importante:** o `Regiao` entra como feature, então parte do desempenho pode vir apenas das **diferenças estruturais entre regiões** (por exemplo, o Norte ter historicamente abandono mais alto), e não do mercado de trabalho. A importância por permutação ajuda a separar esses efeitos.

### 8.3 Limitações

- **Poucas observações** (região × ano, em torno de 45): os resultados são sensíveis e o teste é pequeno. Métricas devem ser lidas como **indicativas**, não definitivas.
- **Dados agregados:** não permitem concluir que o trabalho de um jovem causou seu abandono (falácia ecológica). A análise é de **associação e previsão, não de causalidade**.
- **Lacunas de dados:** anos sem informação do IBGE na base integrada ficam fora da modelagem, e a informalidade ausente nos primeiros anos foi imputada pela mediana.
- **Mudança de período:** o teste (pós-pandemia) tem dinâmica diferente do treino, o que dificulta a generalização.
- **Rede neural:** com tão poucos dados, não é a escolha mais indicada; sua inclusão atende ao enunciado e serve de comparação.

### 8.4 Próximos passos possíveis

Incluir mais variáveis (renda, escolaridade dos pais, infraestrutura escolar), usar dados por **estado** (mais observações), separar as faixas etárias de 14–17 e 18–24 anos como features distintas e testar validação temporal com janelas deslizantes.

## 9. Conclusão

Transformamos a pergunta descritiva da Parte 1 em um problema preditivo de **regressão**, comparando um modelo do scikit-learn (**Random Forest**), um de árvores com *boosting* (**XGBoost**) e uma **rede neural profunda (Keras)**, todos contra um **baseline** e avaliados em anos não vistos com **MAE, RMSE e R²**. As conclusões numéricas desta execução estão na seção 8.1; a interpretação sobre o que isso acrescenta à Parte 1 está na seção 8.2. A principal contribuição desta etapa é medir, e não apenas supor, se os indicadores de mercado de trabalho juvenil ajudam a antecipar o abandono escolar, deixando claros os limites dessa evidência.

## 10. Referências e fontes oficiais

- INEP — Taxas de Rendimento Escolar: https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/indicadores-educacionais/taxas-de-rendimento-escolar/
- IBGE — SIDRA, Tabela 4094: https://apisidra.ibge.gov.br/values/t/4094/n2/all/v/4099,4723,12466,4090/p/201304,201404,201504,201604,201704,201804,201904,202004,202104,202204,202304,202404,202504/c58/114535,100052/h/y/f/a/d/m
- Scikit-learn: https://scikit-learn.org/ | XGBoost: https://xgboost.readthedocs.io/ | Keras: https://keras.io/
- **Data de acesso às fontes:** 26/08/2026